# Genie Cache & Queue — Demo

A API do Genie tem um hard limit de **5 chamadas por minuto por workspace**.
Este notebook demonstra o problema e como o app resolve com cache semantico, retry e fila.

## Cenarios:
1. **Sem o app**: 7 chamadas diretas ao Genie → ultimas recebem **HTTP 429**
2. **Com o app (1a rodada)**: Mesmas 7 queries via app → todas completam
3. **Com o app (2a rodada)**: Mesmas queries de novo → **todas do cache em <0.1s**

## Setup

**Importante**: O Databricks Apps proxy requer OAuth (nao aceita PAT).
Em notebooks DBR 14+, `WorkspaceClient()` gera OAuth automaticamente.
Se o token nao funcionar com o App, a celula de verificacao avisa.

In [ ]:
import requests
import time
import json

WORKSPACE_HOST = "https://fevm-genie-cache-test.cloud.databricks.com"
APP_HOST = "https://genie-cache-queue-7474650836156271.aws.databricksapps.com"
SPACE_ID = "01f11f1ae00114379e7671c8a4b8459f"
WAREHOUSE_ID = "20cbd6750a2ef9ce"

# Obter token OAuth — tenta SDK primeiro, fallback para dbutils
TOKEN = None
try:
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    _h = w.config.authenticate()
    TOKEN = _h.get('Authorization', '').replace('Bearer ', '')
    print(f"Auth: SDK ({w.config.auth_type})")
except Exception as e1:
    print(f"SDK auth falhou: {e1}")

if not TOKEN:
    try:
        TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        print("Auth: dbutils token")
    except:
        import os
        TOKEN = os.getenv("DATABRICKS_TOKEN", "")
        print("Auth: env var")

token_type = "JWT/OAuth" if TOKEN.startswith("eyJ") else "PAT" if TOKEN.startswith("dapi") else "Internal"
print(f"Token type: {token_type}")

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

questions = [
    "How many customers are there?",
    "What is the total revenue by region?",
    "Top 5 suppliers by order count",
    "Average order value per year",
    "How many parts are in the catalog?",
    "Total revenue for EUROPE region",
    "Number of orders in 1995",
]

def extract(data):
    sql, text = None, None
    for att in data.get("attachments", []):
        if isinstance(att, dict):
            if "query" in att and att["query"]:
                sql = att["query"].get("query", "")
            if "text" in att and att["text"]:
                t = att["text"].get("content", "")
                if t and "semantic cache" not in t:
                    text = t
    if not sql:
        sql = data.get("sql_query")
    return sql, text

print(f"\nQuestions: {len(questions)}")

### Verificar autenticacao com o App

In [ ]:
# Verificar se o token funciona com a API do workspace (Genie direto)
r1 = requests.get(f"{WORKSPACE_HOST}/api/2.0/sql/warehouses", headers=headers)
print(f"Workspace API: {r1.status_code} {'OK' if r1.status_code == 200 else r1.text[:80]}")

# Verificar se o token funciona com o App
r2 = requests.get(f"{APP_HOST}/api/v1/health", headers=headers)
app_ok = r2.status_code == 200 and r2.text.strip() != '{}'
print(f"App API:       {r2.status_code} {r2.text[:80]}")

if not app_ok:
    print("\n" + "=" * 60)
    print("TOKEN NAO FUNCIONA COM O APP")
    print("O Apps proxy requer OAuth JWT (nao aceita PAT/internal token).")
    print("")
    print("Opcoes:")
    print("1. Rode este notebook localmente com:")
    print("   export DATABRICKS_TOKEN=$(databricks auth token --profile GENIE-CACHE-OAUTH | jq -r .access_token)")
    print("2. Ou use DBR 14+ com OAuth nativo")
    print("=" * 60)
else:
    r3 = requests.get(f"{APP_HOST}/api/v1/debug/headers", headers=headers)
    debug = r3.json() if r3.status_code == 200 else {}
    h = debug.get('headers', {})
    print(f"App email:     {h.get('x-forwarded-email', 'N/A')}")
    print(f"App token:     {'x-forwarded-access-token' in h}")
    print("\nAuth OK — pode prosseguir")

## Cenario 1: Chamadas diretas ao Genie

7 queries diretas. Limite: **5/min por workspace**. As ultimas devem dar **429**.

In [ ]:
print("=" * 90)
print("CENARIO 1: Chamadas DIRETAS ao Genie")
print("=" * 90)

direct_results = []
for i, q in enumerate(questions, 1):
    start = time.time()
    resp = requests.post(
        f"{WORKSPACE_HOST}/api/2.0/genie/spaces/{SPACE_ID}/start-conversation",
        headers=headers, json={"content": q}, timeout=120,
    )
    elapsed = time.time() - start
    http = resp.status_code
    data = resp.json() if resp.text.strip() else {}
    sql, text = extract(data)

    if http == 429:
        print(f"[{i}/7] 429 RATE LIMITED | {elapsed:.1f}s")
        print(f"       Query:     {q}")
        print(f"       Resultado: BLOQUEADO — rate limit excedido")
    elif sql:
        print(f"[{i}/7] 200 OK | {elapsed:.1f}s")
        print(f"       Query:     {q}")
        print(f"       SQL:       {sql[:100]}")
        if text:
            print(f"       Resposta:  {text[:140]}")
    else:
        print(f"[{i}/7] {http} | {elapsed:.1f}s")
        print(f"       Query:     {q}")
        print(f"       Raw:       {json.dumps(data)[:200]}")

    direct_results.append({"q": q, "http": http, "t": elapsed, "sql": sql, "text": text})
    print()

ok = sum(1 for r in direct_results if r["sql"])
blocked = sum(1 for r in direct_results if r["http"] == 429)
print(f"RESULTADO: {ok} OK com SQL, {blocked} bloqueadas por rate limit")

## Cenario 2: Via App (cache + retry + fila)

Mesmas 7 queries via app. **Mesmo endpoint, so muda a URL base.**

### Configurar app

In [ ]:
resp = requests.put(f"{APP_HOST}/api/v1/config", headers=headers,
    json={"genie_space_id": SPACE_ID, "sql_warehouse_id": WAREHOUSE_ID,
          "similarity_threshold": 0.92, "cache_ttl_seconds": 86400})
print("Config:", json.dumps(resp.json(), indent=2))

### 2a. Primeira rodada (popula cache)

Queries novas. App chama Genie, gerencia rate limit, faz retry. **Caller nunca ve 429.**

In [ ]:
print("=" * 90)
print("CENARIO 2a: Via APP — primeira rodada")
print("=" * 90)

app1 = []
for i, q in enumerate(questions, 1):
    start = time.time()
    resp = requests.post(
        f"{APP_HOST}/api/2.0/genie/spaces/{SPACE_ID}/start-conversation",
        headers=headers, json={"content": q}, timeout=180,
    )
    elapsed = time.time() - start
    data = resp.json() if resp.status_code == 200 and resp.text.strip() not in ('', '{}') else {}

    if not data:
        print(f"[{i}/7] HTTP {resp.status_code} | {elapsed:.1f}s")
        print(f"       Query: {q}")
        print(f"       Erro:  {resp.text[:150]}")
        app1.append({"q": q, "t": elapsed, "cache": False, "sql": None, "text": None})
        print()
        continue

    sql, text = extract(data)
    conv = data.get("conversation_id", "")
    cached = conv.startswith("ccache_")
    status = data.get("status", "?")
    src = "CACHE" if cached else "GENIE"

    print(f"[{i}/7] {status} | {src} | {elapsed:.1f}s")
    print(f"       Query:     {q}")
    if sql:
        print(f"       SQL:       {sql[:100]}")
    if text:
        print(f"       Resposta:  {text[:140]}")
    if not sql and not text:
        print(f"       Raw:       {json.dumps(data)[:200]}")

    app1.append({"q": q, "t": elapsed, "cache": cached, "sql": sql, "text": text})
    print()

genie = sum(1 for r in app1 if not r["cache"] and r["sql"])
cached = sum(1 for r in app1 if r["cache"])
print(f"RESULTADO: {genie} via Genie, {cached} do cache, 0 erros 429 para o caller")

### 2b. Segunda rodada (cache hit)

Mesmas queries — todas do cache. **SQL identico**, resposta em <0.1s.

In [ ]:
print("=" * 90)
print("CENARIO 2b: Via APP — segunda rodada (cache)")
print("=" * 90)

app2 = []
for i, q in enumerate(questions, 1):
    start = time.time()
    resp = requests.post(
        f"{APP_HOST}/api/2.0/genie/spaces/{SPACE_ID}/start-conversation",
        headers=headers, json={"content": q}, timeout=120,
    )
    elapsed = time.time() - start
    data = resp.json() if resp.status_code == 200 else {}
    sql, text = extract(data)
    conv = data.get("conversation_id", "")
    cached = conv.startswith("ccache_")

    prev_sql = app1[i-1]["sql"] if i-1 < len(app1) else None
    match = "MATCH" if sql and prev_sql and sql.strip() == prev_sql.strip() else ("-" if not sql else "NEW")

    print(f"[{i}/7] {'CACHE' if cached else 'GENIE'} | {elapsed:.3f}s | SQL: {match}")
    print(f"       Query: {q}")
    if sql:
        print(f"       SQL:   {sql[:100]}")

    app2.append({"q": q, "t": elapsed, "cache": cached, "sql": sql})
    print()

total = sum(r["t"] for r in app2)
all_cached = all(r["cache"] for r in app2)
print(f"RESULTADO: {sum(1 for r in app2 if r['cache'])}/7 cache hits | Total: {total:.2f}s | Media: {total/7:.3f}s")
print(f"Todas do cache: {all_cached}")

### Cache entries

In [ ]:
resp = requests.get(f"{APP_HOST}/api/v1/cache", headers=headers)
entries = resp.json() if resp.status_code == 200 else []
print(f"Cache entries: {len(entries)}\n")
for e in entries:
    print(f"  Query:   {e['query_text'][:60]}")
    print(f"  SQL:     {e['sql_query'][:80]}")
    print(f"  Uses:    {e['use_count']}")
    print()

## Comparacao

In [ ]:
print(f"{'Query':30s} | {'Direto':>14s} | {'App 1a':>16s} | {'App 2a':>14s}")
print("-" * 80)

for i, q in enumerate(questions):
    d = direct_results[i] if i < len(direct_results) else {"http": "?", "t": 0, "sql": None}
    a1 = app1[i] if i < len(app1) else {"t": 0, "cache": False, "sql": None}
    a2 = app2[i] if i < len(app2) else {"t": 0, "cache": False}

    d_col = "429 BLOCKED" if d["http"] == 429 else f"{d['t']:.1f}s OK" if d["sql"] else f"{d['http']}"
    a1_col = f"{a1['t']:.1f}s CACHE" if a1.get("cache") else f"{a1['t']:.1f}s GENIE" if a1.get("sql") else f"{a1['t']:.1f}s ?"
    a2_col = f"{a2['t']:.2f}s CACHE" if a2.get("cache") else f"{a2['t']:.2f}s ?"

    print(f"  {q[:28]:28s} | {d_col:>14s} | {a1_col:>16s} | {a2_col:>14s}")

print()
print("Direto:  queries 6-7 bloqueadas (429 rate limit)")
print("App 1a:  todas completam (retry automatico)")
print("App 2a:  todas do cache (<0.1s, mesmo SQL)")